In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from transformers import T5TokenizerFast, CLIPProcessor, CLIPTokenizerFast, CLIPImageProcessorFast, T5ForConditionalGeneration


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [ ]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            VLM_CHECKPOINT_DIR,
                            T5_DECODER_LORA_CONFIG)

from Modules.FusionVLM import FusionVLM, create_default_FusionVLM, load_default_FusionVLM, save_FusionVLM
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats, add_dict
from Modules.metrics import evaluate_captioning, setup_nltk
from Modules.train import train_and_evaluate_model

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [6]:
CLIP_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True, local_files_only=True)
# CLIP_tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME, local_files_only=True)
collator = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)

In [7]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [8]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [ ]:
# import numpy as np
# from torch.utils.data import DataLoader, Subset

# num_train_samples = 1024
# num_test_samples = 128


# indices = np.random.choice(len(train_dataset), num_train_samples, replace=False)
# train_subset = Subset(train_dataset, indices)
# train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

# indices = np.random.choice(len(test_dataset), num_test_samples, replace=False)
# test_subset = Subset(test_dataset, indices)
# test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [7]:
model = create_default_FusionVLM().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {num_params:,}\nText Decoder", end=' ')
# model.text_decoder.print_trainable_parameters()

c:\Users\Mahan\Documents\Projects\Retrieval-Augmented-Image-Captioning\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [8]:
from Modules.FusionVLM import apply_lora_config

In [9]:
model = FusionVLM(vision_encoder_name=CLIP_MODEL_NAME,
                    text_encoder_name=T5_MODEL_NAME,
                    T5_text_decoder_name=T5_MODEL_NAME,
                    num_fusion_blocks=4,
                    use_local_files=True
                    )


In [10]:
model = apply_lora_config(model).to(DEVICE)

In [11]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
vision_proj                                   590,592      590,592            0
text_proj                                     590,592      590,592            0
fusion_blocks                              56,724,480   56,724,480            0
post_fusion_ln                                  1,536        1,536            0
fusion_proj                                   590,592      590,592            0
text_decoder                              254,655,744   31,752,192  222,903,552
--------------------------------------------------------------------------------
TOTAL                                     510,238,080   90,249,984  419,988,096


In [ ]:
print_model_param_stats(model)

In [ ]:
NUM_EPOCHS = 2
full_history = {}
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
os.makedirs(VLM_CHECKPOINT_DIR, exist_ok=True)

NameError: name 'model' is not defined

In [17]:
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer)
add_dict(full_history, history)

Epoch 1: 100%|██████████| 1924/1924 [16:13<00:00,  1.98it/s, loss=2.33]


BLEU-1: 0.4330
BLEU-2: 0.2621
BLEU-3: 0.1591
BLEU-4: 0.0961
METEOR: 0.3343
ROUGE-L: 0.3282
CIDEr: 0.1285


Epoch 2: 100%|██████████| 1924/1924 [16:40<00:00,  1.92it/s, loss=2.58]


BLEU-1: 0.4626
BLEU-2: 0.2806
BLEU-3: 0.1690
BLEU-4: 0.1015
METEOR: 0.3468
ROUGE-L: 0.3433
CIDEr: 0.1764


In [18]:
save_FusionVLM(model, f'epoch2', VLM_CHECKPOINT_DIR)

In [19]:
full_history

{'train_loss': [[2.9281836694838352, 2.6275224209823134]],
 'test_loss': [[2.514082007937961, 2.4279847599211193]],
 'BLEU-1': [[0.433043829529388, 0.4626252669623788]],
 'BLEU-2': [[0.2621002485371701, 0.28056511548257757]],
 'BLEU-3': [[0.15914298292913795, 0.16895240909449022]],
 'BLEU-4': [[0.09614547536731834, 0.10146776378624518]],
 'METEOR': [[np.float64(0.3343139875965573), np.float64(0.3467532871924209)]],
 'ROUGE-L': [[np.float64(0.3281914177719219), np.float64(0.3433161412525547)]],
 'CIDEr': [[np.float64(0.12851122134358278), np.float64(0.17638351750005743)]]}

In [20]:
import json
with open('history.json', "w", encoding="utf-8") as f:
    json.dump(full_history, f, indent=2, ensure_ascii=False)

In [ ]:
with torch.no_grad():
    for batch in test_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=1,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.8,
            # repetition_penalty=1.2,
        )

        decoded = T5_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for i in range(len(decoded)):
            print(gt_captions[i])
            print(decoded[i])            
        break

['A man is in an enclosed outdoor swimming area and diving off the edge into a pool of clear blue water .', 'A man wearing a swimsuit dives into a pool surrounded by a screen .', 'A male is diving into a swimming pool in a natural setting .', 'A man  diving into the pool  is wearing a swimsuit .', 'A boy in a swimming suit is diving into a pool .']
A picture ofpooling in the pool pool.
['An old bearded man with long gray and light brown hair wearing a black cowboys hat  a black t-shirt  with a blue  white and red guitar strap around his head  standing in front of a microphone singing .', 'An older male wearing a black cowboy hat in front of a microphone .', 'An older white man in a black cowboy hat sings into a microphone .', 'Willie Nelson looking stage right while at the microphone .', 'Willie Nelson in a black shirt and cowboy hat .']
A picture of a man and his wife is a picture of a man and his wife.
['An older man with a gray braided beard holds his stomach while talking .', 'An o